# DeepSpot-H: from a slide to embeddings, on your own machine

**The slide stays here. What this notebook writes is one file of numbers, and
the last cells open that file and show you everything in it.**

DeepSpot-H is the foundation model for H&E images. It turns each tile of your
slide into one embedding. Run it on your own machine and no image is uploaded:
you keep the slide, and you decide afterwards whether the numbers go anywhere
at all.

| Step | What happens | Where it runs |
|---|---|---|
| 1 | fetch one published slide | here |
| 2 | open it and read its resolution out of the file | here |
| 3 | ask the service which model and which encoder | three calls, none about your slide |
| 4 | cut the tiles and judge each one | here |
| 5 | run DeepSpot-H over them | here |
| 6 | open the file it wrote and check what is in it | here |
| 7 | see the tissue in the embeddings | here |

The other half of a prediction, turning those embeddings into spatial gene
expression, runs on our service. [deepspot-m-quickstart][m] picks up from the
file this notebook writes.

[m]: https://github.com/auroraomics/deepspot-m-quickstart

## Before you start

```bash
pip install -r ../requirements.txt
auroraomics login you@institution.edu
```

The key is for step 3 and nothing else, and what those calls carry is spelled
out there. [Get a key][access] if you do not have one; a request is read by a
person, so ask before you need it.

[access]: https://docs.auroraomics.org/get-access/


---
## 0. Check the install

Versions, and the folder this notebook is reading from. If the import fails,
`pip install -r requirements.txt` has not run in this kernel's environment. The
`embed` extra is not checked here; step 5 is the first thing that needs it, and
the package names the missing extra itself rather than failing obscurely.


In [ ]:
import subprocess
import sys
from importlib import metadata
from pathlib import Path

assert sys.version_info >= (3, 10), f"Python 3.10 or newer is needed; this is {sys.version.split()[0]}"

# The folder this notebook lives in, whichever directory the kernel started in.
HERE = Path.cwd() if (Path.cwd() / "example_slide.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(HERE))

import matplotlib.pyplot as plt
import numpy as np

import auroraomics as ao
from auroraomics import contracts, pack

print("notebook in ", HERE)
print("python      ", sys.version.split()[0])
print("auroraomics ", metadata.version("auroraomics"))
print("numpy       ", np.__version__)


---
## 1. One published slide

A lung-cancer section from an open-access dataset:

> Dawo, S., Nonchev, K., and Silina, K. (2025). *10x Visium Spatial
> Transcriptomics Dataset: Kidney (3) and Lung (5) Cancer with Tertiary
> Lymphoid Structures*. Zenodo. <https://doi.org/10.5281/zenodo.14620362>

Published under **CC BY 4.0**. You may use it, including commercially, as long
as you credit the authors above. The cell below prints that citation every time
it runs, because a file on disk with no provenance is how attribution gets lost.

The dataset is published as one two-gigabyte archive of eight slides and their
assays, and one slide out of it is what this notebook needs. `example_slide.py`
reads the archive's own member directory over the network and then pulls that
one member out by its byte span: five requests, about 210 MB, and the member's
checksum is verified before anything is written. To run on your own slide
instead, point `SLIDE` at it and delete the last line of this cell. Nothing
after this cell knows which slide it got.


In [ ]:
from example_slide import CITATION, LICENCE, SLIDE_MEMBER, fetch_member

DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
SLIDE = DATA / "LC1.tif"

print("Example data:", CITATION)
print("Licence:     ", LICENCE)
print()

fetch_member(SLIDE_MEMBER, SLIDE)


---
## 2. Open it, and read its resolution

A tile is a physical size in micrometres, so a picture with no scale has no
tiles. `open_slide` reads the resolution out of the file, and **refuses a file
that records none** rather than assuming one: a tile declared at the wrong
scale gives a wrong prediction and no error at all. If your own slide records
no resolution, pass the number yourself with `ao.open_slide(path, mpp=...)`.

Only the square being read is decoded, so a slide far larger than memory works.


In [ ]:
slide = ao.open_slide(SLIDE)
height, width = slide.shape[:2]

print(slide)
print(f"{width:,} x {height:,} px at {slide.mpp:.4f} um/px")
print(f"which is {width * slide.mpp / 1000:.1f} x {height * slide.mpp / 1000:.1f} mm of tissue")


In [ ]:
def overview(slide, max_edge=900):
    """A small picture of the whole slide, read one band of rows at a time."""
    height, width = slide.shape[:2]
    step = max(1, -(-max(height, width) // max_edge))
    band = step * 128
    return np.concatenate(
        [
            np.asarray(slide[y : min(y + band, height), 0:width])[::step, ::step]
            for y in range(0, height, band)
        ],
        axis=0,
    )


small = overview(slide)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(small)
ax.set_title(f"the section: {width:,} x {height:,} px at {slide.mpp:.3f} um/px", fontsize=10)
ax.set_axis_off()
plt.show()


---
## 3. Ask the service what the model reads

Three answers come from the service, and all three end up recorded in the file
you write, so a reader of that file can tell what made it:

- **the model card** states the tile size in micrometres. Take it from the card
  rather than typing it, and a slide cut today still matches the model
  tomorrow.
- **the encoder** is the one the model was served with, pinned to one exact
  published version of its weights. This is what makes your embeddings
  comparable with anyone else's; computing them against something else produces
  a file that looks identical and means something different.
- **the quality floors** are the service's, pooled across every model, so your
  machine and the service judge a tile the same way.

Two of these three calls carry your key; the quality floors take none. Not one
of them says anything about your slide: no image, no file name, no metadata.


In [ ]:
with ao.Client() as client:
    cards = client.models()
    registry = client.encoders()
    thresholds = client.qc_thresholds()

card = next(c for c in cards if "embeddings" in c["accepts"]["observations"])
model_id = card["id"]
patch_um = float(card["input_spec"]["patch_um"])
encoder = ao.resolve_encoder(registry)     # the one this model was served with

print("model         ", model_id)
print("it reads      ", ", ".join(card["accepts"]["observations"]))
print("tile size     ", patch_um, "um")
print("encoder       ", encoder.name)
print("quality floors", thresholds.as_dict())


---
## 4. Where the tiles fall, and which ones are kept

The tile size is the model's and the resolution is the slide's, so the pixel
edge is one division. `pack.tile_edge_px` is the one place that division is
written, and both the cutting and the checking read it, so a file cut here
cannot fail a check there.

Every candidate tile is then judged before a model sees it: enough tissue, in
focus, actually stained. The verdict is yours and it happens here, before
DeepSpot-H runs. The count it leaves is what the run below will cost, so it is
worth looking at first.


In [ ]:
edge = pack.tile_edge_px(patch_um, slide.mpp)
cols, rows = width // edge, height // edge

candidates = [
    (int((i + 0.5) * edge), int((j + 0.5) * edge))
    for j in range(rows)
    for i in range(cols)
]

verdicts = np.array(
    [ao.qc_tile(pack.tile_at(slide, cx, cy, edge), thresholds).passes for cx, cy in candidates]
)
positions = np.array(candidates)

print(f"one tile is {patch_um:g} um, which on this slide is {edge} px")
print(f"a regular grid is {cols} x {rows} = {len(candidates):,} candidate tiles")
print(f"{int(verdicts.sum()):,} pass the quality floors, {int((~verdicts).sum()):,} are dropped")
print(f"one submission may carry at most {contracts.max_items('embeddings'):,} rows")


In [ ]:
def on_the_grid(x, y, step, values, background=1.0):
    """Lay one value per tile out on the grid the tiles were cut on."""
    col = np.rint((x - x.min()) / step).astype(int)
    row = np.rint((y - y.min()) / step).astype(int)
    canvas = np.full((row.max() + 1, col.max() + 1) + np.shape(values)[1:], background, dtype=float)
    canvas[row, col] = values
    return canvas


kept_map = on_the_grid(
    positions[:, 0],
    positions[:, 1],
    edge,
    np.where(verdicts[:, None], [0.12, 0.37, 0.75], [0.93, 0.93, 0.93]),
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.4))
axes[0].imshow(small)
axes[0].set_title("the section", fontsize=10)
axes[1].imshow(kept_map, interpolation="nearest")
axes[1].set_title(
    f"one pixel per candidate tile: {int(verdicts.sum()):,} kept, {int((~verdicts).sum()):,} dropped",
    fontsize=10,
)
for ax in axes:
    ax.set_axis_off()
plt.show()


In [ ]:
rng = np.random.default_rng(0)
picks = rng.choice(np.flatnonzero(verdicts), size=6, replace=False)

fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax, i in zip(axes, picks):
    cx, cy = candidates[i]
    ax.imshow(pack.tile_at(slide, cx, cy, edge))
    ax.set_title(f"({cx}, {cy})", fontsize=8)
    ax.set_axis_off()
fig.suptitle(f"six kept tiles: {edge} px each, which is {patch_um:g} um of tissue", fontsize=10)
plt.show()


---
## 5. Run DeepSpot-H

### First, one command before the hours

`doctor` runs DeepSpot-H once over a reference image shipped inside the
package, compares the result against this model's own reference, and times the
pass. That answers the two questions worth asking before a long run: would
numbers from this machine be accepted, and what does one tile cost here.
`--tiles` turns that into a wall clock for this slide.

**It runs on a CPU.** A GPU is used when there is one. Without one the run is
slower and it still runs.

`--allow-download` lets it fetch the pinned weights if this machine does not
already hold them. They come once from wherever the service pins them and are
then cached here; every later run is offline. Drop the flag to be told what is
missing instead of fetching it.


In [ ]:
wanted = [c for c, ok in zip(candidates, verdicts) if ok]

subprocess.run(
    ["auroraomics", "doctor", "--model", model_id, "--tiles", str(len(wanted)), "--allow-download"],
    check=False,
)


### Then the run

`embed_tiles` walks the tiles one batch at a time. It reads, judges and
prepares the next batch while the current one is being encoded, and it holds
one batch of pixels rather than a slide's worth, so this runs on a laptop.

Every value it needs is already in hand and every one of them came from
somewhere: `mpp` from the slide file, `patch_um` from the model card, the
encoder and the floors from the service. Nothing below is typed from memory.

To try a corner of the slide before committing to all of it, set `REGION` to a
window in full-resolution pixels.


In [ ]:
REGION = None                # or (x0, y0, x1, y1) in full-resolution pixels
OUT = DATA / "LC1-embeddings.npz"

if REGION is not None:
    x0, y0, x1, y1 = REGION
    wanted = [(cx, cy) for cx, cy in wanted if x0 <= cx < x1 and y0 <= cy < y1]

print(f"embedding {len(wanted):,} tiles into {OUT}")

report = ao.embed_tiles(
    slide,
    OUT,
    mpp=slide.mpp,             # read out of the slide file
    patch_um=patch_um,         # read off the model card
    encoder=encoder,           # the one the service pins
    thresholds=thresholds,     # the service's quality floors
    centers=wanted,
)

print(f"{report.rows:,} rows of {report.dim} numbers, from {report.considered:,} candidates")
print("dropped by:", report.rejected_by_reason or "nothing")
print("encoder:   ", report.encoder.name)
print("tile edge: ", report.patch_px, "px")


---
## 6. What is in the file

This is the cell a data-governance review would run. The container contract
says which members an embeddings file has; the file on disk is compared against
it, member by member, and every member is printed with its shape and its type.

Three things per tile: the embedding, the tile's centre on the slide, and the
quality measurements that kept it. One more for the file as a whole: a
calibration embedding, computed from a fixed image shipped inside the package
through the same encoder as your rows, so a reader can tell what produced the
file before reading a single row of it.

No tile, no overview image, no slide label, no scanner metadata, no file name.


In [ ]:
written = np.load(OUT, allow_pickle=False)
contract = contracts.embeddings_npz()

print(f"{OUT} — {OUT.stat().st_size / 1e6:.1f} MB\n")
print(f"{'member':20} {'shape':14} type")
for name in sorted(written.files):
    value = written[name]
    print(f"{name:20} {str(value.shape):14} {value.dtype}")

missing = [name for name in contract["required"] if name not in written.files]
extra = [name for name in written.files if name not in contract["keys"] + contract["attrs"]]
print("\nrequired by the contract and missing:", missing or "none")
print("present and not named by the contract:", extra or "none")


---
## 7. See the tissue in the embeddings

An embedding is a row of a few hundred numbers and no one can read one. Two
pictures say whether the run did what it claims.

**A map.** Reduce every embedding to three numbers, paint those as a colour and
lay each tile back where it came from. Nothing here knows anything about the
slide, so any structure you recognise came out of the embeddings.

**Neighbours.** Take one tile and find the five whose embeddings are closest to
it. Tissue that looks alike should come back together.


In [ ]:
emb = written["emb"].astype(np.float32)
x, y, step = written["px_col"], written["px_row"], int(written["patch_px"])

centred = emb - emb.mean(axis=0)
_, _, axes_of_variation = np.linalg.svd(centred, full_matrices=False)
scores = centred @ axes_of_variation[:3].T

low, high = np.percentile(scores, [2, 98], axis=0)
colour = np.clip((scores - low) / np.maximum(high - low, 1e-9), 0, 1)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(on_the_grid(x, y, step, colour), interpolation="nearest")
ax.set_title(f"{len(emb):,} tiles, coloured by their embeddings", fontsize=10)
ax.set_axis_off()
plt.show()


In [ ]:
unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)
query = int(np.argmax(scores[:, 0]))
closest = np.argsort(-(unit @ unit[query]))[:6]

fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for rank, (ax, i) in enumerate(zip(axes, closest)):
    ax.imshow(pack.tile_at(slide, int(x[i]), int(y[i]), step))
    ax.set_title("query" if rank == 0 else f"#{rank}", fontsize=9)
    ax.set_axis_off()
fig.suptitle("one tile, and the five whose embeddings are closest to it", fontsize=10)
plt.show()

slide.close()


---
## 8. What left this machine, and what is next

Nothing of the slide left. What did go out: two calls carrying your key and
nothing else (which model, which encoder), one carrying nothing at all (the
quality floors), and, the first time only, the pinned weights arriving on this
machine. None of that said anything about your slide. The image, its label, its
file metadata and every tile cut from it are still here, and the file you wrote
holds exactly the members you just listed.

**Keeping the embeddings.** The file is a result on its own: one row per tile,
with the position each row came from. Cluster it, retrieve against it, or train
on it, with no further involvement from us.

**Going on to gene expression.** Submitting that file is the only step that
sends anything, and it is still no image.
[deepspot-m-quickstart][m] starts from this file: DeepSpot-M turns these
embeddings into spatial gene expression, and it runs on our service on both
routes.

[m]: https://github.com/auroraomics/deepspot-m-quickstart

| | |
|---|---|
| What DeepSpot-H is | <https://docs.auroraomics.org/models/deepspot-h/> |
| This route, written out | <https://docs.auroraomics.org/guides/embeddings/> |
| What each dropped tile means | <https://docs.auroraomics.org/check-tiles/> |
| Quotas and size caps | <https://docs.auroraomics.org/limits/> |
| Privacy and retention | <https://docs.auroraomics.org/privacy/> |
